<a href="https://colab.research.google.com/github/Harp380/ai-cooking-assistant/blob/main/notebook_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍳 AI Cooking Assistant
### MLH Global Hack Week — Season Kickoff

A personalised meal suggester that learns your preferences over time.

**What we're building:**
1. An onboarding flow that builds your cooking profile
2. A meal suggester that reads your profile and asks Gemini for ideas
3. A feedback loop that rates suggestions and improves over time

**Stack:** Python · Pandas · Gemini API · Gradio

---

> **Getting started on Google Colab:**
> 1. Click the 🔑 key icon in the left sidebar
> 2. Add a secret named `GEMINI_API_KEY`
> 3. Paste your key from [aistudio.google.com](https://aistudio.google.com) (free, no credit card)
> 4. Enable notebook access for the secret
> 5. Run cells from top to bottom following along with the stream


## 0. Setup

Install dependencies and configure your free Gemini API key.

Get yours free at [aistudio.google.com](https://aistudio.google.com) — sign in with Google, click **Get API key**. No credit card needed.


In [1]:
!pip install gradio google-generativeai pandas -q

In [ ]:
import gradio as gr
import google.generativeai as genai
import pandas as pd
import os
from pathlib import Path

# --- API Key Setup ---
# On Google Colab: uses the secrets manager (recommended)
# Locally: paste your key directly into the string below

try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
    print("✅ Loaded API key from Colab secrets")
except Exception:
    API_KEY = ""  # TODO: paste your key here if running locally
    print("Running locally — paste your API key above")

if not API_KEY:
    print("⚠️  No API key found. Add it to Colab secrets or paste it above.")
else:
    genai.configure(api_key=API_KEY)
    model = genai.GenerativeModel("gemini-1.5-flash")
    print("✅ Gemini ready")

## 1. The Profile CSV

Before we build the UI, let's think about what we're storing.

Every user has a profile that tracks:
- Their kitchen equipment
- Dietary restrictions and allergies
- Time they typically have available
- Flavour preferences
- Past meal suggestions and how they rated them

This is what makes suggestions get better over time — the prompt we send Gemini gets richer with every session.

> **Note on Colab:** the CSV lives in your Colab session storage and resets when your session ends. That's fine for this workshop — in a real app you'd save to Google Drive or a database.


In [ ]:
# Profile file locations
PROFILE_PATH = Path("data/profile.csv")
HISTORY_PATH = Path("data/history.csv")

# Create data directory if it doesn't exist
PROFILE_PATH.parent.mkdir(exist_ok=True)

# TODO: write a load_profile() function that:
# - checks if PROFILE_PATH exists
# - if it does, reads it with pd.read_csv and returns the first row as a dict
# - if it doesn't, returns an empty dict {}
def load_profile():
    pass

# TODO: write a save_profile(profile) function that:
# - takes a dict called profile
# - saves it to PROFILE_PATH as a CSV using pd.DataFrame([profile]).to_csv()
def save_profile(profile: dict):
    pass

# TODO: write a load_history() function that:
# - checks if HISTORY_PATH exists
# - if it does, reads and returns it with pd.read_csv
# - if it doesn't, returns an empty DataFrame with columns ["meal", "rating", "date"]
def load_history():
    pass

# TODO: write a save_to_history(meal, rating) function that:
# - loads existing history
# - creates a new row with the meal, rating, and today's date (pd.Timestamp.now().strftime("%Y-%m-%d"))
# - appends it with pd.concat and saves back to HISTORY_PATH
def save_to_history(meal: str, rating: int):
    pass

print("Profile functions ready ✅")
print(f"Existing profile: {load_profile()}")

## 2. Onboarding UI

The first time someone uses the app, they answer a series of questions. This gets saved to their profile CSV and used to build smarter prompts.

We're using **Gradio** to build the UI — it creates a web interface directly from Python with no HTML or CSS needed.


In [ ]:
# TODO: write a save_onboarding function that takes these arguments:
# equipment, dietary, allergies, time_available, spice_level, cuisine_prefs
# - builds a profile dict from these values
# - calls save_profile() to save it
# - returns a confirmation string like "✅ Profile saved!"
def save_onboarding(equipment, dietary, allergies, time_available, spice_level, cuisine_prefs):
    pass

# The Gradio UI is built for you — your job is to wire up the save_onboarding function
with gr.Blocks(title="AI Cooking Assistant — Setup") as onboarding_ui:
    gr.Markdown("## 👋 Let's set up your cooking profile")
    gr.Markdown("Answer these once and we'll personalise every suggestion to your kitchen.")

    equipment = gr.CheckboxGroup(
        choices=["Hob", "Oven", "Air fryer", "Microwave", "Instant Pot", "BBQ"],
        label="What cooking equipment do you have?",
        value=["Hob", "Oven"]
    )

    dietary = gr.CheckboxGroup(
        choices=["None", "Vegetarian", "Vegan", "Pescatarian", "Halal", "Kosher"],
        label="Dietary requirements",
        value=["None"]
    )

    allergies = gr.Textbox(
        label="Any allergies? (e.g. nuts, dairy, gluten)",
        placeholder="Leave blank if none",
    )

    time_available = gr.Slider(
        minimum=10, maximum=120, value=30, step=5,
        label="How many minutes do you usually have to cook?"
    )

    spice_level = gr.Radio(
        choices=["Mild", "Medium", "Hot", "Extra hot"],
        label="Spice preference",
        value="Medium"
    )

    cuisine_prefs = gr.CheckboxGroup(
        choices=["Italian", "Asian", "Mexican", "Indian", "Middle Eastern", "British", "American", "Mediterranean"],
        label="Favourite cuisines",
        value=["Italian", "Asian"]
    )

    save_btn = gr.Button("Save my profile →", variant="primary")
    output = gr.Textbox(label="Status")

    # TODO: wire up the button click to your save_onboarding function
    # save_btn.click(fn=..., inputs=[...], outputs=output)

# share=True is required on Colab
onboarding_ui.launch(share=True)

## 3. Building the Prompt

This is the most interesting part — how you structure the prompt determines the quality of the suggestion.

We build a prompt that includes:
- The user's full profile
- Their past highly-rated meals (so Gemini knows what they like)
- Meals they didn't enjoy (so Gemini avoids them)
- What they have available tonight

**This is prompt engineering** — the context you give the LLM matters as much as the question itself.


In [ ]:
# TODO: write a build_prompt(tonight_info, profile, history) function that:
# - gets top rated meals from history (rating >= 4)
# - gets avoided meals from history (rating <= 2)
# - builds and returns a prompt string that includes:
#   * the user's equipment, dietary needs, allergies, time, spice level, cuisine prefs
#   * their top rated meals
#   * their avoided meals
#   * tonight_info (what they have available / their situation)
# - asks Gemini to suggest ONE specific meal with: name, why it suits them,
#   key ingredients, time to cook, how to start
def build_prompt(tonight_info: str, profile: dict, history) -> str:
    pass

# Test it — once you've written the function, run this to see what gets sent to Gemini
test_profile = {
    "equipment": "Hob, Oven",
    "dietary": "None",
    "allergies": "nuts",
    "time_available": 30,
    "spice_level": "Medium",
    "cuisine_prefs": "Italian, Asian"
}

print("=== PROMPT SENT TO GEMINI ===\n")
print(build_prompt(
    "I have chicken, some pasta, and about 35 minutes",
    test_profile,
    load_history()
))

## 4. The Main App

Now we wire everything together — the user describes their situation tonight, we build the prompt from their profile, call Gemini, and show the suggestion.

They can then rate it, which gets saved to history and improves future prompts.


In [ ]:
# Store the last suggestion so we can rate it
last_suggestion = {"text": ""}

# TODO: write a get_suggestion(tonight_info) function that:
# - returns an error message if tonight_info is empty
# - loads the profile, returns an error if no profile found
# - loads history
# - calls build_prompt() to build the prompt
# - calls model.generate_content(prompt) to get a suggestion
# - stores the suggestion text in last_suggestion["text"]
# - returns the suggestion text and gr.update(visible=True) for the rating row
# - catches exceptions and returns the error message
def get_suggestion(tonight_info: str):
    pass

# TODO: write a rate_suggestion(rating) function that:
# - returns an error if last_suggestion["text"] is empty
# - calls save_to_history() with the first 100 chars of the suggestion and the rating
# - returns a confirmation string
def rate_suggestion(rating: int):
    pass

with gr.Blocks(title="AI Cooking Assistant") as main_ui:
    gr.Markdown("## 🍳 What should I cook tonight?")

    tonight = gr.Textbox(
        label="What's your situation tonight?",
        placeholder="e.g. I've got chicken, some leftover rice, and about 40 minutes. Pretty tired so nothing too complicated.",
        lines=3
    )

    suggest_btn = gr.Button("Suggest something →", variant="primary")

    suggestion_output = gr.Textbox(
        label="Tonight's suggestion",
        lines=10,
        interactive=False
    )

    with gr.Column(visible=False) as rating_row:
        gr.Markdown("### How does that sound?")
        with gr.Row():
            rating = gr.Slider(minimum=1, maximum=5, value=3, step=1, label="Rate this suggestion (1-5)")
            rate_btn = gr.Button("Save rating")
        rating_status = gr.Textbox(label="", interactive=False)

    # TODO: wire up suggest_btn.click to get_suggestion
    # inputs: [tonight], outputs: [suggestion_output, rating_row]

    # TODO: wire up rate_btn.click to rate_suggestion
    # inputs: [rating], outputs: [rating_status]

# share=True required on Colab
main_ui.launch(share=True)

## 5. Your Cooking History

See how your profile is building up over time.


In [ ]:
# TODO: print your profile and history using load_profile() and load_history()
# Show each key/value from the profile
# Show the full history dataframe


## 6. Mini Challenge (10 mins)

Pick one to extend the app:

1. **Ingredients tracker** — add a fridge inventory to onboarding, update it after each meal suggestion
2. **Weekly planner** — modify the prompt to ask Gemini for 5 meals for the week instead of just tonight
3. **Nutritional goal** — add a dietary goal (high protein, low carb etc) to the profile and include it in the prompt
4. **Cuisine roulette** — add a "Surprise me" button that picks a random cuisine outside their usual preferences

Share your Colab link in the chat when you're done! 🚀
